### 1. Importación del Dataframe de Incendios


In [ ]:
import pandas as pd
import numpy as np

df_incendios = pd.read_csv('Dataset/fires-all.csv')
df_provincias = pd.read_excel('Dataset/tabla_secundaria.xlsx')
print(df_incendios.columns)
print(df_incendios.info())

### 2. Limpieza de Datos

In [ ]:
df_incendios.isnull().sum()

In [ ]:
# Borramos las columnas con nulos
df_incendios = df_incendios.drop(columns=['lat', 'lng', 'latlng_explicit', 'causa_supuesta'])
display(df_incendios.head())

In [ ]:
# Vamos a ver cuantas veces aparece ideterminado como municipio
len(df_incendios[df_incendios['municipio'] == 'INDETERMINADO'])

Convertimos los municipios indeterminados en Null

In [ ]:
df_incendios['municipio'] = df_incendios['municipio'].replace('INDETERMINADO', np.nan)
print(len(df_incendios[df_incendios['municipio'] == 'INDETERMINADO']))
df_incendios.head()

Quitamos espacios en blanco a los municipios

In [ ]:
df_incendios['municipio'] = df_incendios['municipio'].str.strip()

Vamos a comprobar duplicados

In [ ]:
total_duplicados = df_incendios.duplicated().sum()
print(total_duplicados)

Vamos a separar las fechas en día, mes y año

In [ ]:
# Aseguramos que la columna sea de tipo fecha
df_incendios['fecha'] = pd.to_datetime(df_incendios['fecha'])

# Creamos las nuevas columnas
df_incendios['dia'] = df_incendios['fecha'].dt.day
df_incendios['mes'] = df_incendios['fecha'].dt.month
df_incendios['anio'] = df_incendios['fecha'].dt.year

In [ ]:
df_incendios.head()

### 3. Limpieza de la Tabla Provincias

In [ ]:
display(df_provincias.head())

In [ ]:
# Renombramos las columnas
df_provincias = df_provincias.rename(columns={
    'CPRO': 'idprovincia',
    'CODAUTO': 'idcomunidad',
    'Comunidad Autónoma': 'comunidad_autonoma',
    'Provincia': 'provincia'
})

In [ ]:
df_provincias.info()

Vamos a quitar las barras de las provincias y quedarnos con la parte en castellano

In [ ]:
# Limpiamos las barras en provincias (ej. "Alicante/Alacant" -> "Alicante")
df_provincias['provincia'] = df_provincias['provincia'].str.split('/').str[0].str.strip()

# Hacemos lo mismo con comunidad por si hubiera algún caso similar
df_provincias['comunidad_autonoma'] = df_provincias['comunidad_autonoma'].str.split('/').str[0].str.strip()

In [ ]:
# Eliminamos los espacios en blanco
df_provincias['provincia'] = df_provincias['provincia'].str.strip()
df_provincias['comunidad_autonoma'] = df_provincias['comunidad_autonoma'].str.strip()

In [ ]:
df_provincias.sample(5)

### 4. Limpieza de la Tabla Causas

In [ ]:
df_causas = pd.read_excel('Dataset/tabla_secundaria.xlsx', sheet_name=1)

In [ ]:
df_causas

Vamos a renombrar la columna id por causaid

In [ ]:
# Renombramos las columnas
df_causas = df_causas.rename(columns={
    'id': 'idcausa'
})
# Eliminamos la columna en inglés
df_causas = df_causas.drop(columns=['en'])

In [ ]:
df_causas.head()

In [ ]:
# Eliminamos los espacios en blanco
df_causas['es'] = df_causas['es'].str.strip()

df_causas = df_causas.rename(columns={'es': 'causa'})

Vamos a cambiar el nombre de las columnas causas y causas_desc de incendios

In [ ]:
df_incendios = df_incendios.rename(columns={
    'causa': 'idcausa',
    'causa_desc': 'idcausa_desc'
})

In [ ]:
df_incendios.head()

### 5. Limpieza de la tabla Causa_Desc

In [ ]:
df_causas_desc = pd.read_excel('Dataset/tabla_secundaria.xlsx', sheet_name=2)
df_causas_desc.head()

In [ ]:
# Eliminamos filas que estén completamente vacías (típico en Excels)
df_causas_desc = df_causas_desc.dropna(subset=['idcausa_desc'])
df_causas_desc['idcausa_desc'] = df_causas_desc['idcausa_desc'].astype(int)

In [ ]:
# Limpiamos los espacios en blanco
df_causas_desc['causa_desc'] = df_causas_desc['causa_desc'].str.strip()

In [ ]:
df_causas_desc

In [ ]:
# Vamos a limpiar nulos
df_causas_desc = df_causas_desc.dropna(subset=['idcausa_desc', 'causa_desc'])

Vamos a unificar la tabla

In [ ]:
# Agrupamos por ID y unimos las descripciones únicas
df_causas_desc = df_causas_desc.groupby('idcausa_desc')['causa_desc'].apply(lambda x: ' para '.join(dict.fromkeys(x))).reset_index()

# Limpieza final de espacios por si acaso
df_causas_desc['causa_desc'] = df_causas_desc['causa_desc'].str.strip()

In [ ]:
# Filtramos para ver los resultados de los IDs específicos
comprobacion = df_causas_desc[df_causas_desc['idcausa_desc'].isin([221, 223, 222])]
display(comprobacion)

### 6. Creación de la Base de Datos

1. Definición del Esquema Relacional

In [ ]:
import sqlite3

conexion = sqlite3.connect('incendios.db')
cursor = conexion.cursor()

# Habilitamos el soporte de Foreign Keys en SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

# Tabla de Provincias (Dimensión)
cursor.execute('''
CREATE TABLE provincias (
    idprovincia INTEGER PRIMARY KEY,
    provincia TEXT,
    idcomunidad INTEGER,
    comunidad_autonoma TEXT
)
''')

# Tabla de Causas Generales (Dimensión)
cursor.execute('''
CREATE TABLE causas (
    idcausa INTEGER PRIMARY KEY,
    causa TEXT
)
''')

# Tabla de Causas Detalladas (Dimensión)
cursor.execute('''
CREATE TABLE causas_desc (
    idcausa_desc INTEGER PRIMARY KEY,
    causa_desc TEXT
)
''')

# Tabla Principal de Incendios (Hechos) con las relaciones
cursor.execute('''
CREATE TABLE IF NOT EXISTS incendios (
    id INTEGER PRIMARY KEY,
    superficie REAL,
    fecha TEXT,
    idprovincia INTEGER,
    municipio TEXT,
    idcausa INTEGER,
    idcausa_desc INTEGER,
    muertos INTEGER,
    heridos INTEGER,
    time_ctrl INTEGER,
    time_ext INTEGER,
    personal INTEGER,
    medios INTEGER,
    gastos REAL,
    perdidas REAL,
    dia INTEGER,
    mes INTEGER,
    anio INTEGER,
    FOREIGN KEY (idprovincia) REFERENCES provincias (idprovincia),
    FOREIGN KEY (idcausa) REFERENCES causas (idcausa),
    FOREIGN KEY (idcausa_desc) REFERENCES causas_desc (idcausa_desc)
    
)
''')

conexion.commit()

2. Carga de datos

In [ ]:
# 1. Insertamos los datos en las tablas maestras (Provincias, Causas, Causas_desc)
df_provincias.to_sql('provincias', conexion, if_exists='append', index=False)
df_causas.to_sql('causas', conexion, if_exists='append', index=False)
df_causas_desc.to_sql('causas_desc', conexion, if_exists='append', index=False)

# 2. Definimos la lista completa de columnas que ya tienes en tu DataFrame
columnas_principales = [
    'id', 'superficie', 'fecha', 'idprovincia', 'municipio', 'idcausa', 'idcausa_desc', 
    'muertos', 'heridos', 'time_ctrl', 'time_ext', 'personal', 'medios', 'gastos', 
    'perdidas', 'dia', 'mes', 'anio'
]

# 3. Insertamos el DataFrame de incendios con todas sus columnas de impacto y tiempo
df_incendios[columnas_principales].to_sql('incendios', conexion, if_exists='append', index=False)

# 4. Cerramos la conexión
conexion.close()

In [ ]:
# Comprobamos si hay IDs de provincias en incendios que no estén en la maestra de provincias
missing_provincias = df_incendios[~df_incendios['idprovincia'].isin(df_provincias['idprovincia'])]
print(f"IDs de provincias faltantes: {missing_provincias['idprovincia'].unique()}")

# Comprobamos causas
missing_causas = df_incendios[~df_incendios['idcausa'].isin(df_causas['idcausa'])]
print(f"IDs de causas faltantes: {missing_causas['idcausa'].unique()}")

# Comprobamos causas detalladas
missing_desc = df_incendios[~df_incendios['idcausa_desc'].isin(df_causas_desc['idcausa_desc'])]
print(f"IDs de causa_desc faltantes: {missing_desc['idcausa_desc'].unique()}")